In [1]:
import os
import numpy as np
import rasterio
from pysheds.grid import Grid
import numpy
print(numpy.__version__)

import numba
print("numba OK")

from pysheds.grid import Grid
print("pysheds OK")

downloads_dir = os.path.expanduser("~/Downloads")
candidates = [f for f in os.listdir(downloads_dir) if "dem" in f.lower() and f.lower().endswith((".tif", ".tiff"))]

if not candidates:
    raise FileNotFoundError(f"No DEM file found in {downloads_dir}. Files present: {os.listdir(downloads_dir)}")

dem_path = os.path.join(downloads_dir, candidates[0])
print(f"Using DEM file: {dem_path}")

with rasterio.open(dem_path) as src:
    print("Full raster size:", src.width, "x", src.height)
    print("CRS:", src.crs)
    print("dtype:", src.dtypes)
    print(f"Approx full-array size in memory (float32): {src.width * src.height * 4 / 1e9:.2f} GB")

    win_size = 500
    cx, cy = src.width // 2, src.height // 2
    window = rasterio.windows.Window(
        cx - win_size // 2, cy - win_size // 2, win_size, win_size
    )
    test_dem = src.read(1, window=window)
    test_transform = src.window_transform(window)

    profile = src.profile.copy()
    profile.update({
        "height": win_size,
        "width": win_size,
        "transform": test_transform,
    })
    test_path = "dem_sanity_test_crop.tif"
    with rasterio.open(test_path, "w", **profile) as dst:
        dst.write(test_dem, 1)

print(f"Saved {win_size}x{win_size} test crop to {test_path}")

grid = Grid.from_raster(test_path)
dem = grid.read_raster(test_path)

print("Shape:", dem.shape)
print("Min/Max elevation:", np.nanmin(dem), np.nanmax(dem))
print("NaN fraction:", np.isnan(dem).mean())

pit_filled_dem = grid.fill_pits(dem)
flooded_dem = grid.fill_depressions(pit_filled_dem)
inflated_dem = grid.resolve_flats(flooded_dem)

fdir = grid.flowdir(inflated_dem)
print("Unique fdir values:", np.unique(fdir))

acc = grid.accumulation(fdir)
print("Acc min/max/mean:", acc.min(), acc.max(), acc.mean())
print("Acc stdDev:", np.std(acc))

2.3.5
numba OK
pysheds OK
Using DEM file: C:\Users\user/Downloads\zimbabwe_dem_glo30.tif
Full raster size: 29016 x 25267
CRS: EPSG:4326
dtype: ('float32',)
Approx full-array size in memory (float32): 2.93 GB
Saved 500x500 test crop to dem_sanity_test_crop.tif
Shape: (500, 500)
Min/Max elevation: 1182.4783 1317.1906
NaN fraction: 0.0


C:\Users\user\anaconda3\envs\mineral-prospect\Lib\site-packages\pysheds\io.py:134: UserWarning: No `nodata` value detected. Defaulting to 0.
  warnings.warn('No `nodata` value detected. Defaulting to 0.')
C:\Users\user\anaconda3\envs\mineral-prospect\Lib\site-packages\pysheds\io.py:134: UserWarning: No `nodata` value detected. Defaulting to 0.
  warnings.warn('No `nodata` value detected. Defaulting to 0.')


Unique fdir values: [ -2  -1   1   2   4   8  16  32  64 128]
Acc min/max/mean: 1.0 78468.0 192.619856
Acc stdDev: 2300.450544694786
